In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import os
from pathlib import Path
import sys

CWD = os.getcwd()
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

In [2]:
from src.data import TextDataset
from torch.utils.data import DataLoader

from src.modules import QLoRA
import torch.optim as optim

MODEL_NAME = "Qwen/Qwen3.5-2B"
EPOCH_SIZE = 100
EVAL_ITER = 2
LEARNING_RATE = 1e-3
ACCUMULATION_STEPS = 4

train_data = TextDataset(tokenizer_model=MODEL_NAME)
loader = DataLoader(train_data.data[train_data.data_sources[0]],
            batch_size=1,
            shuffle=True,
            collate_fn=lambda rows: rows[0],
        )
qwen_qlora = QLoRA(model_name=MODEL_NAME, rank=16, alpha=32, device=device)

c:\Users\taput\anaconda3\envs\chatbot-fraudster\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\taput\anaconda3\envs\chatbot-fraudster\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\taput\.cache\huggingface\hub\models--Qwen--Qwen3.5-2B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In orde

In [3]:
params    = [p for n, p in qwen_qlora.named_parameters() if p.requires_grad ]
optimizer = optim.AdamW(params=params, weight_decay=1e-2, lr=LEARNING_RATE)

# Train

In [4]:
for step, batch in enumerate(loader, start=1):
    if step > EPOCH_SIZE:
        break

    batch = {
        key: value.unsqueeze(0).to("cuda:0")
        for key, value in batch.items()
    }

    loss = qwen_qlora(**batch).loss

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}: {loss.item()}")

    (loss / ACCUMULATION_STEPS).backward()
    if step % ACCUMULATION_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    print(f"step {step}: loss={loss.item():.4f}")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


step 1: loss=0.1235
step 2: loss=0.7999
step 3: loss=0.6348
step 4: loss=0.3605
step 5: loss=1.5847
step 6: loss=0.8830
step 7: loss=0.4420
step 8: loss=0.3527
step 9: loss=0.1568
step 10: loss=0.0751
step 11: loss=0.2036
step 12: loss=0.1345
step 13: loss=1.5905
step 14: loss=0.0140
step 15: loss=0.0471
step 16: loss=0.0452
step 17: loss=1.0409
step 18: loss=0.0384
step 19: loss=0.0314
step 20: loss=0.3175
step 21: loss=1.4170
step 22: loss=1.1848
step 23: loss=1.1452
step 24: loss=0.4048
step 25: loss=0.0721
step 26: loss=0.1496
step 27: loss=0.1126
step 28: loss=1.1090
step 29: loss=0.7460
step 30: loss=0.0418
step 31: loss=0.2233
step 32: loss=0.2507
step 33: loss=2.0210
step 34: loss=1.0705
step 35: loss=0.3214
step 36: loss=0.1948
step 37: loss=0.4298
step 38: loss=0.0707
step 39: loss=1.1869
step 40: loss=0.9088
step 41: loss=1.2739
step 42: loss=0.7102
step 43: loss=0.3642
step 44: loss=0.0330
step 45: loss=0.3365
step 46: loss=1.1742
step 47: loss=0.3382
step 48: loss=0.5039
s

In [5]:
adapter_weights = {
    name: parameter.detach().cpu()
    for name, parameter in qwen_qlora.named_parameters()
    if name.endswith(("lora_a", "lora_b"))
}

torch.save(
    {
        "base_model": MODEL_NAME,
        "rank": 16,
        "alpha": 32,
        "weights": adapter_weights,
    },
    "qwen3.5-4b-toolace-lora.pt",
)

In [6]:
# Check which parameters are trainable.
trainable = [
    name for name, p in qwen_qlora.named_parameters()
    if p.requires_grad
]

print("Trainable tensors:", len(trainable))
print("First few:", trainable[:10])

assert all(
    name.endswith(("lora_a", "lora_b"))
    for name in trainable
), "A non-LoRA parameter is trainable"

Trainable tensors: 372
First few: ['model.model.layers.0.linear_attn.out_proj.lora_a', 'model.model.layers.0.linear_attn.out_proj.lora_b', 'model.model.layers.0.linear_attn.in_proj_qkv.lora_a', 'model.model.layers.0.linear_attn.in_proj_qkv.lora_b', 'model.model.layers.0.linear_attn.in_proj_z.lora_a', 'model.model.layers.0.linear_attn.in_proj_z.lora_b', 'model.model.layers.0.linear_attn.in_proj_b.lora_a', 'model.model.layers.0.linear_attn.in_proj_b.lora_b', 'model.model.layers.0.linear_attn.in_proj_a.lora_a', 'model.model.layers.0.linear_attn.in_proj_a.lora_b']


In [8]:
# Save it to Huggingface transformer version:

In [10]:
import json
from pathlib import Path

import torch
from safetensors.torch import save_file

SOURCE = Path("qwen3.5-2b-toolace-lora.pt")
OUTPUT = Path("../models/Qwen/qwen3.5-2b-toolace-adapter")

checkpoint = torch.load(SOURCE, map_location="cpu", weights_only=True)
assert checkpoint["base_model"] == "Qwen/Qwen3.5-2B"

weights = checkpoint["weights"]
converted = {}
targets = set()

for name, tensor in weights.items():
    if name.endswith(".lora_a"):
        module_name = name.removesuffix(".lora_a")
        suffix = "lora_A.weight"
    elif name.endswith(".lora_b"):
        module_name = name.removesuffix(".lora_b")
        suffix = "lora_B.weight"
    else:
        raise ValueError(f"Unexpected checkpoint key: {name}")

    # PEFT-style key: base_model.model.<original module path>.<A or B>
    new_name = f"base_model.model.{module_name}.{suffix}"
    converted[new_name] = tensor.detach().cpu().contiguous()
    targets.add(module_name.split(".")[-1])

assert len(converted) == len(weights)
assert len(converted) % 2 == 0

OUTPUT.mkdir(exist_ok=True)
save_file(converted, OUTPUT / "adapter_model.safetensors")

config = {
    "peft_type": "LORA",
    "task_type": "CAUSAL_LM",
    "base_model_name_or_path": checkpoint["base_model"],
    "r": checkpoint["rank"],
    "lora_alpha": checkpoint["alpha"],
    "lora_dropout": 0.0,
    "bias": "none",
    "target_modules": sorted(targets),
    "inference_mode": True,
}

(OUTPUT / "adapter_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)
print(f"Exported {len(converted) // 2} LoRA pairs")
print("Targets:", sorted(targets))

Exported 186 LoRA pairs
Targets: ['down_proj', 'gate_proj', 'in_proj_a', 'in_proj_b', 'in_proj_qkv', 'in_proj_z', 'k_proj', 'o_proj', 'out_proj', 'q_proj', 'up_proj', 'v_proj']
